# Red Wine Quality — EDA + ML

**Датасет:** `winequality-red.csv` — 1599 красных вин, 11 химических признаков + оценка качества (3–9).

**Цель:** понять что влияет на качество вина и построить модель предсказания.

**Структура:**
1. Первичный осмотр
2. Data Quality Audit
3. Распределения и выбросы
4. Корреляционный анализ
5. Модели (GBM + Ridge)
6. Анализ ошибок модели
7. Итоговый Summary


---
## 0. Импорты

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('✓ OK')


---
## Этап 1 — Первичный осмотр

**Цель:** понять структуру данных до любых вычислений.

In [ ]:
df = pd.read_csv('winequality-red.csv')

print(f'Shape: {df.shape}')
print(f'\ndtypes:')
print(df.dtypes)


**Вывод:** 1599 вин × 12 колонок. Все признаки числовые — химические характеристики вина.
`quality` — целевая переменная, дискретная (оценка дегустатора от 3 до 9).


In [ ]:
display(df.head())

print('\nQuality distribution:')
print(df['quality'].value_counts().sort_index())
print(f'\nMean quality: {df["quality"].mean():.2f}')
print(f'Quality range: {df["quality"].min()} – {df["quality"].max()}')

# Дисбаланс классов
print('\nДоля каждого класса:')
print((df['quality'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')


**Вывод — критическая находка: сильный дисбаланс классов.**
~82% вин получили оценку 5 или 6. Оценки 3 и 8 встречаются крайне редко (< 1% и < 2%).

Это важно для модели: она будет хорошо предсказывать «средние» вина и плохо — экстремальные.
MAE нужно анализировать отдельно по группам качества, а не только в среднем.


---
## Этап 2 — Data Quality Audit

**Цель:** найти проблемы до анализа.

In [ ]:
print('── NaN ────────────────────────────────')
print(df.isna().sum())
print(f'\nДублей строк: {df.duplicated().sum()}')
print(f'\n── Базовая статистика ─────────────────')
print(df.describe().round(2))


**Вывод:** данные чистые — нет NaN. Есть 240 дублей (одинаковые вина измеренные несколько раз) —
это нормально для винного датасета, не удаляем.

Обращаем внимание на:
- `volatile acidity` max = 1.58 (обычно < 0.8 — возможный выброс)
- `total sulfur dioxide` max = 289 (обычно < 150 — возможный выброс)
- `chlorides` max = 0.611 (обычно < 0.2 — возможный выброс)


In [ ]:
# Проверяем конкретные экстремальные значения
print('── volatile acidity > 1.4 ──────────────')
print(df[df['volatile acidity'] > 1.4][['volatile acidity', 'quality']])

print('\n── total sulfur dioxide > 250 ──────────')
print(df[df['total sulfur dioxide'] > 250][['total sulfur dioxide', 'quality']])

print('\n── chlorides > 0.4 ─────────────────────')
print(df[df['chlorides'] > 0.4][['chlorides', 'quality']].head(5))


**Вывод:** выбросы — это реальные вина с экстремальными характеристиками, не ошибки данных.
- Вино с `volatile acidity = 1.58` получило quality = 3 — высокая уксусная кислотность портит вкус ✓
- Вина с `total SO₂ > 250` получили quality = 7 — высокий SO₂ не обязательно плохо ✓

Оставляем все строки — они несут полезную информацию для модели.


---
## Этап 3 — Распределения и выбросы

In [ ]:
numeric_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
                'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
                'pH', 'sulphates', 'alcohol', 'quality']

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', alpha=0.75, edgecolor='none')
    axes[i].axvline(df[col].mean(), color='red', linewidth=1.5, linestyle='--',
                    label=f'mean={df[col].mean():.2f}')
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Распределения всех переменных', fontsize=13)
plt.tight_layout()
plt.show()


**Вывод:**
- Большинство признаков — правосторонние (skewed right): `residual sugar`, `chlorides`, `sulfur dioxide`.
- `alcohol` и `volatile acidity` — ближе к нормальному.
- `quality` — дискретная, похожа на нормальное с центром в 5-6, но сильно несбалансированная.
- Нельзя применять методы, предполагающие нормальность — используем Spearman, не Pearson.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].values, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', markersize=4))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticks([])

plt.suptitle('Boxplot — поиск выбросов', fontsize=13)
plt.tight_layout()
plt.show()


**Вывод:** выбросы есть почти везде, но для вина это нормально —
разные сорта и стили производства дают широкий разброс химических показателей.


---
## Этап 4 — Корреляционный анализ

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

key_features = ['alcohol', 'volatile acidity', 'sulphates',
                'citric acid', 'total sulfur dioxide', 'density']

for i, col in enumerate(key_features):
    corr, p = spearmanr(df[col], df['quality'])
    axes[i].scatter(df[col], df['quality'], alpha=0.15, color='steelblue', s=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('quality')
    axes[i].set_title(f'{col}\nSpearman r={corr:.2f}, p={p:.3f}')

plt.suptitle('Связь ключевых признаков с качеством вина', fontsize=13)
plt.tight_layout()
plt.show()


**Вывод — ключевые корреляции с quality:**
- **alcohol (r>0)** — больше алкоголя = выше качество. Самый сильный предиктор.
- **volatile acidity (r<0)** — больше уксусной кислоты = ниже качество. Логично химически.
- **sulphates (r>0)** — сульфаты как консервант улучшают качество.
- **total sulfur dioxide (r<0)** — избыток SO₂ негативно влияет на вкус.


In [ ]:
corr_matrix = df[numeric_cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            mask=mask, annot_kws={'size': 9})
plt.title('Корреляционная матрица (Spearman) — нижний треугольник', fontsize=12)
plt.tight_layout()
plt.show()

print('── Корреляции с quality (|r|, по убыванию) ──')
corr_q = corr_matrix['quality'].drop('quality')
for feat, val in corr_q.abs().sort_values(ascending=False).items():
    sign = '+' if corr_q[feat] > 0 else '-'
    print(f'  {feat:25s}: {sign}{val:.3f}')


**Вывод:** три главных предиктора качества:
1. `alcohol` — сильная положительная связь (r≈+0.48)
2. `volatile acidity` — сильная отрицательная связь (r≈-0.38)
3. `sulphates` — умеренная положительная связь (r≈+0.38)

Также видна мультиколлинеарность: `fixed acidity` сильно коррелирует с `citric acid` (r=0.66)
и `density` (r=0.62) — это нужно учитывать при интерпретации коэффициентов Ridge.


---
## Этап 5 — Модели

**Валидация:** 5-fold cross-validation — честная оценка без data leakage.

In [ ]:
features = ['fixed acidity', 'volatile acidity', 'citric acid',
            'residual sugar', 'chlorides', 'free sulfur dioxide',
            'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']

X = df[features].copy()
y = df['quality'].copy()

baseline_mae = np.mean(np.abs(y - y.mean()))
print(f'Признаков:     {len(features)}')
print(f'Строк:         {len(X)}')
print(f'Baseline MAE:  {baseline_mae:.2f} (всегда предсказывать среднее)')


**Baseline** — если всегда предсказывать среднее качество (~5.6), MAE ≈ 0.68.
Хорошая модель должна быть заметно лучше этого числа.


In [ ]:
# GBM baseline
gbm_base = GradientBoostingRegressor(n_estimators=100, random_state=42)
scores_gbm = cross_val_score(gbm_base, X, y, cv=5, scoring='r2')
scores_mae_gbm = cross_val_score(gbm_base, X, y, cv=5, scoring='neg_mean_absolute_error')

# Ridge — через Pipeline (scaler честно внутри каждого fold)
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])
scores_ridge = cross_val_score(ridge, X, y, cv=5, scoring='r2')
scores_mae_ridge = cross_val_score(ridge, X, y, cv=5, scoring='neg_mean_absolute_error')

print('── Результаты (5-fold CV) ──────────────────────')
print(f'GBM base R²:   {scores_gbm.mean():.3f} ± {scores_gbm.std():.3f}')
print(f'GBM base MAE:  {-scores_mae_gbm.mean():.3f} балла')
print(f'Ridge    R²:   {scores_ridge.mean():.3f} ± {scores_ridge.std():.3f}')
print(f'Ridge    MAE:  {-scores_mae_ridge.mean():.3f} балла')
print(f'Baseline MAE:  {baseline_mae:.3f}')


In [ ]:
# GridSearch — тюнинг GBM
params = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 4, 5],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0],
}

grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    params, cv=5, scoring='r2', n_jobs=-1, verbose=0
)
grid.fit(X, y)

print(f'Лучший R²:        {grid.best_score_:.3f}')
print(f'Лучшие параметры: {grid.best_params_}')
print(f'Улучшение vs базовый GBM: +{(grid.best_score_ - scores_gbm.mean()):.3f}')


**Вывод:** GridSearch нашёл лучшие параметры. Используем их для финальной модели.
Небольшая глубина (3-4) + медленный learning rate (0.05) + subsample < 1.0 — классика для GBM.


In [ ]:
# Финальная модель с лучшими параметрами из GridSearch
gbm_best = grid.best_estimator_
scores_best = cross_val_score(gbm_best, X, y, cv=5, scoring='r2')
scores_mae_best = cross_val_score(gbm_best, X, y, cv=5, scoring='neg_mean_absolute_error')

print('── Финальное сравнение ─────────────────────────')
results = pd.DataFrame({
    'Model':    ['Baseline', 'Ridge', 'GBM (default)', 'GBM (tuned)'],
    'R²':       [0.0, scores_ridge.mean(), scores_gbm.mean(), scores_best.mean()],
    'MAE':      [baseline_mae, -scores_mae_ridge.mean(),
                 -scores_mae_gbm.mean(), -scores_mae_best.mean()],
}).set_index('Model').round(3)
display(results)

print(f'\nЛучшая модель улучшает baseline на: {(baseline_mae - (-scores_mae_best.mean()))/baseline_mae*100:.0f}%')


**Как читать метрики:**
- **R²** = 0.34 → модель объясняет 34% дисперсии оценок. Для субъективной оценки вина — хорошо.
- **MAE** = 0.49 → в среднем ошибаемся на 0.49 балла из 10.
- R² выше 0.40 для вина считается отличным — оценки субъективны и варьируются между дегустаторами.


In [ ]:
# Feature Importance — из tuned модели
gbm_best.fit(X, y)
ridge.fit(X, y)

fi_gbm = pd.Series(gbm_best.feature_importances_, index=features).sort_values(ascending=True)
fi_ridge = pd.Series(
    np.abs(ridge.named_steps['model'].coef_), index=features
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

fi_gbm.plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('GBM (tuned) Feature Importance')
axes[0].set_xlabel('Важность')

fi_ridge.plot(kind='barh', ax=axes[1], color='coral', alpha=0.8)
axes[1].set_title('Ridge |Коэффициенты|')
axes[1].set_xlabel('|Коэффициент|')

plt.suptitle('Что влияет на качество вина?', fontsize=12)
plt.tight_layout()
plt.show()

print(f'GBM главный предиктор:   {fi_gbm.idxmax()}')
print(f'Ridge главный предиктор: {fi_ridge.idxmax()}')


**Вывод:** обе модели согласны — топ-3 одинаковые:
1. **alcohol** — главный предиктор с большим отрывом
2. **sulphates** — второй по важности
3. **volatile acidity** — третий

Это подтверждает корреляционный анализ. Химически логично: алкоголь определяет
тело и стиль вина, volatile acidity — уксусный привкус, sulphates — свежесть.


---
## Этап 6 — Анализ ошибок модели

**Цель:** понять где модель ошибается — это важнее чем средний MAE.

In [ ]:
# Получаем предсказания через CV (честно — не обучаем на тесте)
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(gbm_best, X, y, cv=5)
y_pred = np.clip(y_pred, y.min(), y.max())

residuals = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Predicted vs Actual
ax = axes[0]
ax.scatter(y, y_pred, alpha=0.2, s=15, color='steelblue')
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1.5, label='Идеал')
ax.set_xlabel('Реальное качество')
ax.set_ylabel('Предсказанное качество')
ax.set_title(f'Predicted vs Actual\nR² = {r2_score(y, y_pred):.3f}  MAE = {mean_absolute_error(y, y_pred):.3f}')
ax.legend()

# 2. Распределение ошибок
ax = axes[1]
ax.hist(residuals, bins=40, color='steelblue', alpha=0.75, edgecolor='none')
ax.axvline(0, color='red', linewidth=1.5, linestyle='--')
ax.axvline(residuals.mean(), color='orange', linewidth=1.5,
           label=f'mean error = {residuals.mean():.3f}')
ax.set_xlabel('Ошибка (реальное − предсказанное)')
ax.set_ylabel('Кол-во вин')
ax.set_title('Распределение ошибок')
ax.legend()

# 3. MAE по группам качества — КЛЮЧЕВОЙ ГРАФИК
ax = axes[2]
mae_by_quality = {}
for q in sorted(y.unique()):
    mask = y == q
    if mask.sum() > 0:
        mae_by_quality[q] = mean_absolute_error(y[mask], y_pred[mask])

bars = ax.bar(mae_by_quality.keys(), mae_by_quality.values(),
              color='coral', alpha=0.8)
ax.axhline(mean_absolute_error(y, y_pred), color='red',
           linewidth=1.5, linestyle='--', label=f'Overall MAE = {mean_absolute_error(y, y_pred):.2f}')
ax.set_xlabel('Quality (реальная оценка)')
ax.set_ylabel('MAE')
ax.set_title('MAE по группам качества\n(где модель ошибается больше?)')
ax.legend()

# Добавляем count на столбцы
for bar, q in zip(bars, mae_by_quality.keys()):
    n = (y == q).sum()
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={n}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


**Вывод — самый важный анализ:**

- **Predicted vs Actual:** точки концентрируются в диапазоне 5-7 — там где больше данных.
  Предсказания "схлопываются" к центру: модель редко предсказывает 3 или 8.
  
- **Распределение ошибок:** симметрично вокруг нуля — нет систематического смещения.

- **MAE по группам (критично):** модель ошибается значительно больше на редких классах:
  - Оценка 3 (очень плохое вино, n < 10): высокая ошибка — мало примеров для обучения
  - Оценки 5-6 (80% данных): низкая ошибка — именно здесь модель хороша
  - Оценка 8 (отличное вино): высокая ошибка — мало примеров

  **Вывод:** средний MAE = 0.49 лестит модели. На крайних оценках ошибка значительно выше.
  Для production нужна либо балансировка классов, либо переформулировка как классификация.


In [ ]:
# Топ-10 самых неудачных предсказаний
errors_df = df.copy()
errors_df['pred'] = y_pred
errors_df['error'] = np.abs(y - y_pred)

print('── Топ-10 крупнейших ошибок ────────────────────')
cols_show = ['quality', 'pred', 'error', 'alcohol', 'volatile acidity', 'sulphates']
display(errors_df.nlargest(10, 'error')[cols_show].round(2))

print('\n── MAE по группам quality ──────────────────────')
for q, mae in mae_by_quality.items():
    n = (y == q).sum()
    print(f'  Quality {q} (n={n:3d}): MAE = {mae:.3f}')


**Вывод:** большие ошибки часто у вин с нетипичным сочетанием признаков —
например, высокий alcohol но низкое качество (возможно, из-за высокой volatile acidity).
Это ограничение модели: она хорошо выучила «средние» паттерны, но плохо обобщает на крайние случаи.


---
## Этап 7 — Итоговый Summary

### Часть 1: Что за данные
1599 красных вин, 11 химических признаков, оценка качества 3–9.
Данные чистые: нет NaN, дубли нормальны для датасета.
**Критически несбалансированный** таргет: 82% вин — оценка 5 или 6.

### Часть 2: Что нашли
- **Главный предиктор — alcohol** (r≈+0.48). Больше алкоголя = выше качество.
- **volatile acidity отрицательно влияет** (r≈-0.38). Уксусная кислотность портит вкус.
- **sulphates положительно влияет** (r≈+0.38). Консервант улучшает свежесть.
- Мультиколлинеарность: fixed acidity ↔ citric acid ↔ density.
- Все распределения правосторонние → Spearman корректный выбор.

### Часть 3: Результаты модели

| Модель | R² | MAE |
|---|---|---|
| Baseline (среднее) | 0.00 | 0.68 |
| Ridge (Pipeline) | ~0.29 | ~0.54 |
| GBM (default) | ~0.32 | ~0.50 |
| **GBM (tuned)** | **~0.34** | **~0.49** |

**Главное ограничение:** MAE = 0.49 — средняя цифра. На редких оценках (3, 8) модель ошибается значительно сильнее из-за дисбаланса классов.

### Что дальше
1. **Балансировка классов** — oversampling редких оценок (SMOTE) или class_weight.
2. **Классификация** — переформулировать как low/medium/high, метрика F1-macro.
3. **Feature engineering** — отношение alcohol/volatile acidity как новый признак.
4. **Объединить с white wine** — больше данных улучшит предсказание крайних классов.


In [ ]:
print('=' * 55)
print('ИТОГОВЫЙ ОТЧЁТ — Red Wine Quality')
print('=' * 55)
print(f'  Вин в датасете:        {len(df)}')
print(f'  Признаков:             {len(features)}')
print(f'  NaN:                   0')
print(f'  Дублей:                {df.duplicated().sum()}')
print(f'  Quality range:         {df["quality"].min()}–{df["quality"].max()}')
print(f'  Дисбаланс (5+6):       {((df["quality"].isin([5,6])).mean()*100):.0f}%')
print()
print(f'  Baseline MAE:          {baseline_mae:.3f}')
print(f'  GBM tuned MAE:         {-scores_mae_best.mean():.3f}')
print(f'  GBM tuned R²:          {scores_best.mean():.3f}')
print(f'  Улучшение vs baseline: {(baseline_mae-(-scores_mae_best.mean()))/baseline_mae*100:.0f}%')
print()
print(f'  Главный предиктор:     alcohol')
print(f'  Слабое место:          редкие классы (3, 8) — мало данных')
print('=' * 55)
